# DANDI 001754 ThreeDimSpatial → NeuroPy pipeline

Loads locally downloaded [DANDI 001754 — Neurolab ThreeDimSpatial](https://dandiarchive.org/dandiset/001754) NWB files from `ThreeDimSpatial/001754` via registered `DANDI001754NWBDataSessionFormat` / `DataSessionLoader.dandi_nwb_001754_session`, then runs the NeuroPy pipeline (flattened spikes, position QC, task epochs, placefields, decoding).

**Workflow:**
1. **Initialize session** — load NWB → write export cache under `ThreeDimSpatial/export/001754/{Rat}/{ses-id}/`
2. **POSTLOAD** — task epochs, laps, PBE/non-PBE
3. **Pipeline** — `NeuropyPipeline` + computations; saves `basedir/loadedSessPickle.pkl`

**Paths:** `sess.basepath` is the subject folder under `001754/`; export cache is under `ThreeDimSpatial/export/001754/{Rat}/{ses-id}/`.

**Install:** `uv sync --all-extras --python 3.10`

**Note:** The preflight `ecephys`-only NWB has no position data. Use a `behavior+ecephys` file via `NWB_FILENAME`.


In [1]:
%config IPCompleter.use_jedi = False
# %xmode Verbose
# %xmode context
%pdb off
%load_ext autoreload
%autoreload 3

import sys
from pathlib import Path

# required to enable non-blocking interaction:
%gui qt5

import importlib
from copy import deepcopy
from numba import jit
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
# pd.options.mode.dtype_backend = 'pyarrow' # use new pyarrow backend instead of numpy
from attrs import define, field, fields, Factory, make_class
import tables as tb
from datetime import datetime, timedelta

# Pho's Formatting Preferences
import builtins

import IPython
from IPython.core.formatters import PlainTextFormatter
from IPython import get_ipython

from pyphocorehelpers.preferences_helpers import set_pho_preferences, set_pho_preferences_concise, set_pho_preferences_verbose
set_pho_preferences_concise()
# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from pyphocorehelpers.gui.Jupyter.AsyncExecutionHelper import run_async

# BEGIN PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #

## IPython pprint
from pyphocorehelpers.pprint import wide_pprint, wide_pprint_ipython, wide_pprint_jupyter, MAX_LINE_LENGTH
# Override default pprint
builtins.pprint = wide_pprint

ip = get_ipython()

from pyphocorehelpers.ipython_helpers import CustomFormatterMagics

# Register the magic
get_ipython().register_magics(CustomFormatterMagics)

text_formatter: PlainTextFormatter = ip.display_formatter.formatters['text/plain']
text_formatter.max_width = MAX_LINE_LENGTH
text_formatter.for_type(object, wide_pprint_jupyter)


# END PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #

from pyphocorehelpers.print_helpers import get_now_time_str, get_now_day_str
from pyphocorehelpers.indexing_helpers import get_dict_subset

## Pho's Custom Libraries:
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path, file_uri_from_path
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
import pyphocorehelpers.programming_helpers as programming_helpers

# NeuroPy (Diba Lab Python Repo) Loading
# from neuropy import core
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from typing_extensions import TypeAlias
from nptyping import NDArray
import neuropy.utils.type_aliases as types

from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.core.epoch import NamedTimerange, Epoch
from neuropy.core.ratemap import Ratemap
from neuropy.core.session.Formats.BaseDataSessionFormats import DataSessionFormatRegistryHolder, DataSessionFormatBaseRegisteredClass
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters
from neuropy.core.session.Formats.Specific.NWBDataSessionFormat import NWBDataSessionFormatRegisteredClass

from neuropy.utils.matplotlib_helpers import matplotlib_file_only, matplotlib_configuration, matplotlib_configuration_update
from neuropy.core.neuron_identities import NeuronIdentityTable, neuronTypesList, neuronTypesEnum
from neuropy.utils.mixins.AttrsClassHelpers import AttrsBasedClassHelperMixin, serialized_field, serialized_attribute_field, non_serialized_field, custom_define
from neuropy.utils.mixins.HDF5_representable import HDF_DeserializationMixin, post_deserialize, HDF_SerializationMixin, HDFMixin, HDF_Converter

## For computation parameters:
from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.utils.dynamic_container import DynamicContainer
from neuropy.utils.result_context import IdentifyingContext
from neuropy.core.session.Formats.BaseDataSessionFormats import find_local_session_paths
from neuropy.core.user_annotations import UserAnnotationsManager

from pyphocorehelpers.print_helpers import print_object_memory_usage, print_dataframe_memory_usage, print_value_overview_only, DocumentationFilePrinter, print_keys_if_possible, generate_html_string, document_active_variables
from pyphocorehelpers.programming_helpers import metadata_attributes
from pyphocorehelpers.function_helpers import function_attributes
## Pho Programming Helpers:
from pyphocorehelpers.print_helpers import DocumentationFilePrinter, TypePrintMode, print_keys_if_possible, debug_dump_object_member_shapes, print_value_overview_only, document_active_variables
from pyphocorehelpers.programming_helpers import IPythonHelpers, PythonDictionaryDefinitionFormat, MemoryManagement, inspect_callable_arguments, get_arguments_as_optional_dict, GeneratedClassDefinitionType, CodeConversion
from pyphocorehelpers.notebook_helpers import NotebookCellExecutionLogger
from pyphocorehelpers.gui.Qt.TopLevelWindowHelper import TopLevelWindowHelper, print_widget_hierarchy
from pyphocorehelpers.indexing_helpers import reorder_columns, reorder_columns_relative, dict_to_full_array
from pyphocorehelpers.DataStructure.RenderPlots.MatplotLibRenderPlots import MatplotlibRenderPlots

# pyPhoPlaceCellAnalysis:
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import NeuropyPipeline # get_neuron_identities
from pyphoplacecellanalysis.General.Mixins.ExportHelpers import export_pyqtgraph_plot
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_load_session, batch_extended_computations, batch_evaluate_required_computations
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import PipelineSavingScheme # used in perform_pipeline_save
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import PipelineJupyterHelpers, CustomProcessingPhases
from pyphocorehelpers.assertion_helpers import Assert

import pyphoplacecellanalysis.External.pyqtgraph as pg

from pyphocorehelpers.exception_helpers import ExceptionPrintingContext, CapturedException
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_perform_all_plots
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.LongShortTrackComputations import JonathanFiringRateAnalysisResult
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import _find_any_context_neurons
from pyphoplacecellanalysis.General.Batch.runBatch import BatchSessionCompletionHandler # for `post_compute_validate(...)`
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder
from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import AcrossSessionsResults
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.SpikeAnalysis import SpikeRateTrends # for `_perform_long_short_instantaneous_spike_rate_groups_analysis`
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.LongShortTrackComputations import SingleBarResult, InstantaneousSpikeRateGroupsComputation, TruncationCheckingResults # for `BatchSessionCompletionHandler`, `AcrossSessionsAggregator`
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import SplitPartitionMembership
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalPlacefieldGlobalComputationFunctions, DirectionalLapsResult, TrackTemplates, DecoderDecodedEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.RankOrderComputations import RankOrderGlobalComputationFunctions,  RankOrderComputationsContainer, RankOrderResult, RankOrderAnalyses
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import TrackTemplates
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.ComputationFunctionRegistryHolder import ComputationFunctionRegistryHolder, computation_precidence_specifying_function, global_function
from pyphocorehelpers.Filesystem.path_helpers import set_posix_windows
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder, DecodedFilterEpochsResult, SingleEpochDecodedResult

from pyphocorehelpers.assertion_helpers import Assert

# Plotting
# import pylustrator # customization of figures
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
_bak_rcParams = mpl.rcParams.copy()

matplotlib.use('Qt5Agg')
# %matplotlib inline
# %matplotlib auto

# _restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')
_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

import seaborn as sns

# import pylustrator # call `pylustrator.start()` before creating your first figure in code.
from pyphoplacecellanalysis.Pho2D.matplotlib.visualize_heatmap import visualize_heatmap, visualize_heatmap_pyqtgraph # used in `plot_kourosh_activity_style_figure`
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import plot_multiple_raster_plot, plot_raster_plot
from pyphoplacecellanalysis.General.Mixins.DataSeriesColorHelpers import UnitColoringMode, DataSeriesColorHelpers
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import _build_default_tick, build_scatter_plot_kwargs
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.Render2DScrollWindowPlot import Render2DScrollWindowPlotMixin, ScatterItemData
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.SpikeAnalysis import SpikeRateTrends
from pyphoplacecellanalysis.General.Mixins.SpikesRenderingBaseMixin import SpikeEmphasisState
from pyphoplacecellanalysis.General.Model.SpecificComputationParameterTypes import ComputationKWargParameters
from pyphoplacecellanalysis.SpecificResults.PhoDiba2023Paper import PAPER_FIGURE_figure_1_add_replay_epoch_rasters, PAPER_FIGURE_figure_1_full, PAPER_FIGURE_figure_3, main_complete_figure_generations
# from pyphoplacecellanalysis.SpecificResults.fourthYearPresentation import *

# Jupyter Widget Interactive
import ipywidgets as widgets
from IPython.display import display, HTML
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import interactive_pipeline_widget, interactive_pipeline_files
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget, render_colors

from datetime import datetime, date, timedelta
from pyphocorehelpers.print_helpers import get_now_day_str, get_now_rounded_time_str


known_data_session_type_properties_dict = DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict()
active_data_session_types_registered_classes_dict = DataSessionFormatRegistryHolder.get_registry_data_session_type_class_name_dict()

DAY_DATE_STR: str = date.today().strftime("%Y-%m-%d")
DAY_DATE_TO_USE = f'{DAY_DATE_STR}' # used for filenames throught the notebook
print(f'DAY_DATE_STR: {DAY_DATE_STR}, DAY_DATE_TO_USE: {DAY_DATE_TO_USE}')

NOW_DATETIME: str = get_now_rounded_time_str()
NOW_DATETIME_TO_USE = f'{NOW_DATETIME}' # used for filenames throught the notebook
print(f'NOW_DATETIME: {NOW_DATETIME}, NOW_DATETIME_TO_USE: {NOW_DATETIME_TO_USE}')

def get_global_variable(var_name):
    """ used by `PipelineJupyterHelpers._build_pipeline_custom_processing_mode_selector_widget(...)` to update the notebook's variables """
    return globals()[var_name]
    
def update_global_variable(var_name, value):
    """ used by `PipelineJupyterHelpers._build_pipeline_custom_processing_mode_selector_widget(...)` to update the notebook's variables """
    globals()[var_name] = value

from pyphocorehelpers.gui.Jupyter.simple_widgets import build_global_data_root_parent_path_selection_widget
all_paths = [Path(r'H:\Data'), Path(r'I:\Data'), Path('/media/halechr/BETAMAX1/Data'), Path(r'/home/halechr/FastData'), Path('/Volumes/SwapSSD/Data'), Path('/Users/pho/data'), Path(r'/media/halechr/MAX/Data'), Path(r"H:\Data"), Path(r'W:\Data'), Path(r'/home/halechr/cloud/turbo/Data'), Path(r'/Volumes/MoverNew/data'), Path(r'/home/halechr/turbo/Data'), Path(r'/Users/pho/cloud/turbo/Data')] # Path('/Volumes/FedoraSSD/FastData'), 
global_data_root_parent_path = None
def on_user_update_path_selection(new_path: Path):
    global global_data_root_parent_path
    new_global_data_root_parent_path = new_path.resolve()
    global_data_root_parent_path = new_global_data_root_parent_path
    print(f'global_data_root_parent_path changed to {global_data_root_parent_path}')
    assert global_data_root_parent_path.exists(), f"global_data_root_parent_path: {global_data_root_parent_path} does not exist! Is the right computer's config commented out above?"
            
global_data_root_parent_path_widget = build_global_data_root_parent_path_selection_widget(all_paths, on_user_update_path_selection)
global_data_root_parent_path_widget

Automatic pdb calling has been turned OFF


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\utils\mixins\time_slicing.py:405: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class TimePointEventAccessor(TimeColumnAliasesProtocol, TimeSlicableObjectProtocol, DataframeMetadataProtocol):
h:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\.venv\lib\site-packages\replay_trajectory_classification\likelihoods\multiunit_likelihood.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
h:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\.venv\lib\site-packages\hdf5storage\utilities.py:44: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_reso

field.name: "merged_directional_placefields", variable_name: "merged_directional_placefields"
field.name: "rank_order_shuffle_analysis", variable_name: "rank_order_shuffle_analysis"
field.name: "directional_decoders_decode_continuous", variable_name: "directional_decoders_decode_continuous"
field.name: "directional_decoders_evaluate_epochs", variable_name: "directional_decoders_evaluate_epochs"
field.name: "directional_decoders_epoch_heuristic_scoring", variable_name: "directional_decoders_epoch_heuristic_scoring"
field.name: "directional_train_test_split", variable_name: "directional_train_test_split"
field.name: "long_short_decoding_analyses", variable_name: "long_short_decoding_analyses"
field.name: "long_short_rate_remapping", variable_name: "long_short_rate_remapping"
field.name: "long_short_inst_spike_rate_groups", variable_name: "long_short_inst_spike_rate_groups"
field.name: "wcorr_shuffle_analysis", variable_name: "wcorr_shuffle_analysis"
field.name: "non_pbe_epochs_results", 

ToggleButtons(description='Data Root:', layout=Layout(width='auto'), options=(WindowsPath('H:/Data'), WindowsPath('I:/Data'), WindowsPath('W:/Data')), style=ToggleButtonsStyle(button_width='max-content'), tooltip='global_data_root_parent_path', value=WindowsPath('H:/Data'))

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

from pynwb import NWBHDF5IO
from neuropy.core.session.data_session_loader import DataSessionLoader
from neuropy.core.session.Formats.BaseDataSessionFormats import DataSessionFormatRegistryHolder
from neuropy.core.session.Formats.Specific.DANDI001754NWBDataSessionFormat import DANDI001754NWBDataSessionFormatRegisteredClass
from neuropy.utils.result_context import IdentifyingContext
from pyphocorehelpers.assertion_helpers import Assert
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import NeuropyPipeline, PipelineSavingScheme
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import final_process_bapun_all_comps, build_non_kdiba_directional_decoders, build_proper_epoch_intervals


In [3]:
# REPO_ROOT = Path(r"H:\Data\DANDI\ThreeDimSpatial").resolve()
REPO_ROOT = global_data_root_parent_path.joinpath("DANDI/ThreeDimSpatial").resolve()

DATA_ROOT = REPO_ROOT / '001754'
EXPORT_ROOT = REPO_ROOT / 'export'


SUBJECT = 'Rat1'
NWB_FILENAME = 'sub-Rat1_ses-19980425T124500_behavior+ecephys.nwb'  # Flight Day 9 (default)
# NWB_FILENAME = 'sub-Rat1_ses-19980420T095700_behavior+ecephys.nwb'  # Flight Day 4

SUBJECT = 'Rat2'
NWB_FILENAME = 'sub-Rat2_ses-19980413T163700_ecephys.nwb'
# NWB_FILENAME = 'sub-Rat2_ses-19980420T095700_behavior+ecephys.nwb'  
# NWB_FILENAME = 'sub-Rat2_ses-19980425T124500_behavior+ecephys.nwb' 



SESSION_BASEDIR = DATA_ROOT / f'sub-{SUBJECT}'
UNIT_LOCATION_FILTER = 'CA1'
PLOT_SUBSAMPLE = 25
active_data_mode_name = 'dandi_nwb_001754'
basedir = SESSION_BASEDIR.resolve()
force_reload = False  # set True to ignore basedir/loadedSessPickle.pkl and rebuild pipeline
saving_mode = PipelineSavingScheme.TEMP_THEN_OVERWRITE

override_parameters_flat_keypaths_dict = {
    'preprocessing.nwb.nwb_filename': NWB_FILENAME,
    'preprocessing.nwb.unit_location_filter': UNIT_LOCATION_FILTER,
    'preprocessing.nwb.export_root': str(EXPORT_ROOT),
    'preprocessing.epoch_estimation_parameters.laps.use_direction_dependent_laps': False,
}



assert DATA_ROOT.is_dir(), f'Missing data root: {DATA_ROOT}'
assert SESSION_BASEDIR.is_dir(), f'Missing subject dir: {SESSION_BASEDIR}'
print('REPO_ROOT:', REPO_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('SESSION_BASEDIR:', SESSION_BASEDIR)
print('NWB_FILENAME:', NWB_FILENAME)
print('active_data_mode_name:', active_data_mode_name)
print('export cache root:', EXPORT_ROOT / '001754' / SUBJECT)


REPO_ROOT: H:\Data\DANDI\ThreeDimSpatial
DATA_ROOT: H:\Data\DANDI\ThreeDimSpatial\001754
SESSION_BASEDIR: H:\Data\DANDI\ThreeDimSpatial\001754\sub-Rat1
NWB_FILENAME: sub-Rat1_ses-19980425T124500_behavior+ecephys.nwb
active_data_mode_name: dandi_nwb_001754
export cache root: H:\Data\DANDI\ThreeDimSpatial\export\001754\Rat1


In [4]:
nwb_files = sorted(SESSION_BASEDIR.glob('*.nwb'))
print(f'Found {len(nwb_files)} NWB files under {SESSION_BASEDIR}')
for p in nwb_files:
    tag = ' (ecephys-only: no position)' if 'behavior' not in p.name else ''
    print(' -', p.name, tag)

nwb_path = DANDI001754NWBDataSessionFormatRegisteredClass.find_nwb_file(SESSION_BASEDIR, nwb_filename=NWB_FILENAME)
print('Selected NWB:', nwb_path)


Found 3 NWB files under H:\Data\DANDI\ThreeDimSpatial\001754\sub-Rat1
 - sub-Rat1_ses-19980414T125300_ecephys.nwb  (ecephys-only: no position)
 - sub-Rat1_ses-19980420T095700_behavior+ecephys.nwb 
 - sub-Rat1_ses-19980425T124500_behavior+ecephys.nwb 
Selected NWB: H:\Data\DANDI\ThreeDimSpatial\001754\sub-Rat1\sub-Rat1_ses-19980425T124500_behavior+ecephys.nwb


In [5]:
with NWBHDF5IO(str(nwb_path), mode='r') as io:
    nwbf = io.read()
    print('session_description:', nwbf.session_description)
    if nwbf.intervals and 'epochs' in nwbf.intervals:
        ep = nwbf.intervals['epochs']
        print('epochs:', len(ep), '| cols:', list(ep.colnames))
        display(ep.to_dataframe())
    if nwbf.processing and 'behavior' in nwbf.processing and 'position' in nwbf.processing['behavior'].data_interfaces:
        ss = nwbf.processing['behavior']['position'].spatial_series['spatial_series']
        print('position shape:', ss.data.shape, '| unit:', ss.unit)
    else:
        print('No behavior position in this NWB file.')
    print('units:', len(nwbf.units) if nwbf.units is not None else 0)
    if nwbf.processing and 'ecephys' in nwbf.processing and 'rate_maps' in nwbf.processing['ecephys'].data_interfaces:
        print('rate_maps rows:', len(nwbf.processing['ecephys']['rate_maps']))


session_description: Flight Day 9 recording — Rat 1. Escher Staircase and Magic Carpet tasks with baseline sessions.
epochs: 4 | cols: ['start_time', 'stop_time', 'session_type', 'session_type_description']


,start_time,stop_time,session_type,session_type_description
id,,,,
0,3178.0,4789.0,ES,Escher Staircase — three-dimensional track wit...
1,4871.0,7535.0,BL,Baseline — rectangular track
2,7888.0,9200.0,MC,Magic Carpet — flat two-dimensional track
3,9220.0,10151.0,BL,Baseline — rectangular track


position shape: (191065, 2) | unit: pixels
units: 39
rate_maps rows: 64


## Initialize session from NWB

First load builds NeuroPy export cache (`.npy` under `export/001754/Rat1/ses-…/`). Re-running uses the cache unless you delete those files or change `NWB_FILENAME`.

In [6]:
# Same load path as the working terminal smoke test:
#   DANDI001754NWBDataSessionFormatRegisteredClass.get_session(basedir, override_parameters_flat_keypaths_dict=...)
sess = DataSessionLoader.dandi_nwb_001754_session(SESSION_BASEDIR, override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict)
print('context:', sess.get_context())
print('neurons:', sess.neurons.n_neurons, '| position samples:', len(sess.position.time))
print('epochs:', sess.epochs.get_unique_labels().tolist())
print('filePrefix:', sess.filePrefix)
for cache_suffix in ['.neurons.npy', '.position.npy', '.paradigm.npy', '.flattened.spikes.npy', '.interpolated_spike_positions.npy']:
    cache_path = sess.filePrefix.with_suffix(cache_suffix)
    print(f'  {cache_path}:', 'OK' if cache_path.exists() else 'MISSING', '->', cache_path)
sess

	 Loading success: .interpolated_spike_positions.npy.
externally computed ripple_df.pkl not found. Falling back to .ripple.npy...
Failure loading .ripple.npy. Must recompute.

computing ripple epochs for session...

Computation failed with error 'NoneType' object has no attribute 'get_signal'. Skipping .ripple
Loading success: .mua.npy.
Loading success: .pbe.npy.
Loading success: .non_pbe.npy.
Computing spikes_df PBEs column results : "spikes_df"... encounter KeyError 't_rel_seconds' when attempting to access spk_df using its spk_df.spikes.time_variable_name variable. Original spk_df.spikes.time_variable_name: "t_rel_seconds". Changing it to "t_seconds" and proceeding forward


	 time variable changed from 't_rel_seconds' to 't_seconds'.


	 time variable changed!
done.
Computing added spike scISI column results : "spikes_df"... done.
context: dandi_nwb_001754_Rat1_001754_ses-19980425T124500
neurons: 39 | position samples: 191065
epochs: ['ES0', 'BL0', 'MC0', 'BL1']
filePrefix: H:\Data\DANDI\ThreeDimSpatial\export\001754\Rat1\ses-19980425T124500
  ses-19980425T124500.neurons.npy: OK -> H:\Data\DANDI\ThreeDimSpatial\export\001754\Rat1\ses-19980425T124500.neurons.npy
  ses-19980425T124500.position.npy: OK -> H:\Data\DANDI\ThreeDimSpatial\export\001754\Rat1\ses-19980425T124500.position.npy
  ses-19980425T124500.paradigm.npy: OK -> H:\Data\DANDI\ThreeDimSpatial\export\001754\Rat1\ses-19980425T124500.paradigm.npy
  ses-19980425T124500.flattened.spikes.npy: OK -> H:\Data\DANDI\ThreeDimSpatial\export\001754\Rat1\ses-19980425T124500.flattened.spikes.npy
  ses-19980425T124500.interpolated_spike_positions.npy: OK -> H:\Data\DANDI\ThreeDimSpatial\export\001754\Rat1\ses-19980425T124500.interpolated_spike_positions.npy


DataSession(configured from manual recinfo: DynamicContainer({'source_file': None, 'channel_groups': None, 'skipped_channels': None, 'discarded_channels': None, 'n_channels': None, 'dat_sampling_rate': None, 'eeg_sampling_rate': None}))

In [7]:
# Epoch/lap/PBE post-load (runs on pipeline init too; safe to run once here after first NWB load)
DANDI001754NWBDataSessionFormatRegisteredClass.POSTLOAD_estimate_laps_and_replays(sess)
print('epochs after POSTLOAD:', sess.epochs.get_unique_labels().tolist())
print('laps:', len(sess.laps.to_dataframe()) if sess.laps is not None else 0)
sess.epochs.to_dataframe()

POSTLOAD_estimate_laps_and_replays()...
fixing up DANDI 001754 session computation epochs...
	done. new epochs: 
5 epochs
array([[0.04, 1611.04],
       [0.04, 6022.04],
       [1693.04, 4357.04],
       [4710.04, 6022.04],
       [6042.04, 6973.04]])


computing PBE epochs for session...

computing estimated replay epochs for session...

	 using KnownFilterEpochs.PBE as surrogate replays...


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\core\session\Formats\Specific\DANDI001754NWBDataSessionFormat.py:411: UserWarning: Could not estimate replays for DANDI 001754 session dandi_nwb_001754_Rat1_001754_ses-19980425T124500: 'DataFrame' object has no attribute 'to_dataframe'
  warnings.warn(f'Could not estimate replays for DANDI 001754 session {sess.get_context()}: {e}')


computing non_PBE epochs for session...



H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\core\epoch.py:2188: UserWarning: curr_epochs already empty prior to any filtering
  warn(f'curr_epochs already empty prior to any filtering')


Saving non_pbe results results : "H:/Data/DANDI/ThreeDimSpatial/export/001754/Rat1/ses-19980425T124500.non_pbe.npy"... ses-19980425T124500.non_pbe.npy saved
done.


DataSession(configured from manual recinfo: DynamicContainer({'source_file': None, 'channel_groups': None, 'skipped_channels': None, 'discarded_channels': None, 'n_channels': None, 'dat_sampling_rate': None, 'eeg_sampling_rate': None}))

epochs after POSTLOAD: ['ES0', 'task_GLOBAL', 'BL0', 'MC0', 'BL1']
laps: 2


,start,stop,label,duration
0,0.04,1611.04,ES0,1611.0
4,0.04,6022.04,task_GLOBAL,6022.0
1,1693.04,4357.04,BL0,2664.0
2,4710.04,6022.04,MC0,1312.0
3,6042.04,6973.04,BL1,931.0


## Load NeuroPy pipeline

Wraps the initialized session for downstream computations. Saves `basedir/loadedSessPickle.pkl` after a successful fresh load.

In [8]:
known_data_session_type_properties_dict = DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict()
active_data_session_types_registered_classes_dict = DataSessionFormatRegistryHolder.get_registry_data_session_type_class_name_dict()
active_data_mode_type_properties = known_data_session_type_properties_dict[active_data_mode_name]

print(f'basedir: {basedir} | force_reload: {force_reload}')
curr_active_pipeline = NeuropyPipeline.try_init_from_saved_pickle_or_reload_if_needed(active_data_mode_name, active_data_mode_type_properties, override_basepath=Path(basedir), force_reload=force_reload, override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict, skip_save_on_initial_load=False)
print('pipeline session context:', curr_active_pipeline.sess.get_context())
print('pickle path:', basedir / 'loadedSessPickle.pkl')


basedir: H:\Data\DANDI\ThreeDimSpatial\001754\sub-Rat1 | force_reload: False
Computing loaded session pickle file results : "H:/Data/DANDI/ThreeDimSpatial/001754/sub-Rat1/loadedSessPickle.pkl"... 	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	added_keys: ['merged_directional_placefields', 'rank_order_shuffle_analysis', 'directional_decoders_decode_continuous', 'directional_decoders_evaluate_epochs', 'directional_decoders_epoch_heuristic_scoring', 'directional_train_test_split', 'long_short_decoding_analyses', 'long_short_rate_remapping', 'long_short_inst_spike_rate_groups', 'wcorr_shuffle_analysis', 'non_pbe_epochs_results', 'position_decoding', 'perform_specific_epochs_decoding', 'DEP_ratemap_peaks', 'ratemap_peaks_prominence2d']
		adding key: merged_directional_placefields
		adding key: rank_order_shuffle_analysis
		adding key: directional_decoders_decode_continuous
		adding key: directional_decoders_evaluate_epochs
		adding 

In [9]:
sess = curr_active_pipeline.sess
epochs_df = sess.epochs.to_dataframe()
pos_df = sess.position.to_dataframe()
print('context:', sess.get_context())
print('neurons:', sess.neurons.n_neurons)
print('position samples:', len(sess.position.time))
print('x range:', float(pos_df['x'].min()), float(pos_df['x'].max()))
print('y range:', float(pos_df['y'].min()), float(pos_df['y'].max()))
display(epochs_df)

fig, ax = plt.subplots(figsize=(8, 8))
sample_df = pos_df.iloc[::PLOT_SUBSAMPLE].copy()
for _, row in epochs_df.iterrows():
    epoch_mask = (sample_df['t'] >= row['start']) & (sample_df['t'] <= row['stop'])
    ax.plot(sample_df.loc[epoch_mask, 'x'], sample_df.loc[epoch_mask, 'y'], '.', ms=1, label=str(row['label']))
ax.set_title('Trajectory by epoch (subsampled)')
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
ax.legend(markerscale=8, fontsize=8)
plt.show()


context: dandi_nwb_001754_Rat1_001754_ses-19980425T124500
neurons: 39
position samples: 191051
x range: 0.0 255.0
y range: 64.0 255.0


,start,stop,label,duration
0,0.04,1611.04,ES0,1611.0
4,0.04,6022.04,task_GLOBAL,6022.0
1,1693.04,4357.04,BL0,2664.0
2,4710.04,6022.04,MC0,1312.0
3,6042.04,6973.04,BL1,931.0


Text(0.5, 1.0, 'Trajectory by epoch (subsampled)')

Text(0.5, 0, 'x (pixels)')

Text(0, 0.5, 'y (pixels)')

In [10]:
with NWBHDF5IO(str(nwb_path), mode='r') as io:
    nwbf = io.read()
    if nwbf.processing and 'ecephys' in nwbf.processing and 'rate_maps' in nwbf.processing['ecephys'].data_interfaces:
        rate_maps = nwbf.processing['ecephys']['rate_maps'].to_dataframe()
        display(rate_maps.head())
        if 'firing_rate' in rate_maps.columns and len(rate_maps) > 0:
            example_map = np.asarray(rate_maps.iloc[0]['firing_rate'])
            plt.figure(figsize=(5, 4))
            plt.imshow(example_map, origin='lower', cmap='viridis')
            plt.title('Example precomputed rate map (unit 0)')
            plt.colorbar(label='Hz')
            plt.show()
    else:
        print('No precomputed rate_maps table in NWB.')


,tetrode,source_file,session_type,cell_number,rate_map,occupancy_map
id,,,,,,
0,TT0,ESCELL~1.RMA,ES,1,"[[-999.0, -999.0, -999.0, -999.0, -999.0, -999...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,TT0,ESCELL~2.RMA,ES,2,"[[-999.0, -999.0, -999.0, -999.0, -999.0, -999...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2,TT0,ESCELL~3.RMA,ES,3,"[[-999.0, -999.0, -999.0, -999.0, -999.0, -999...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
3,TT0,MC2DC0~1.RMA,MC,-1,"[[-999.0, -999.0, -999.0, -999.0, -999.0, -999...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
4,TT0,MC2DC2~1.RMA,MC,-1,"[[-999.0, -999.0, -999.0, -999.0, -999.0, -999...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


In [11]:
DANDI001754NWBDataSessionFormatRegisteredClass.session_fixup_epochs(curr_active_pipeline.sess, enable_global_epoch=True)
curr_active_pipeline = final_process_bapun_all_comps(curr_active_pipeline=curr_active_pipeline, active_data_mode_name=active_data_mode_name, posthoc_save=False, time_bin_size=0.500, overwrite_extant=False, fail_on_exception=True, active_computation_functions_name_includelist=['pf_computation', 'pfdt_computation', 'position_decoding'])


WARN: already fixedup session epochs.
	restoring backed up epochs:
	done. new epochs: 
5 epochs
array([[0.04, 1611.04],
       [0.04, 6022.04],
       [1693.04, 4357.04],
       [4710.04, 6022.04],
       [6042.04, 6973.04]])




     start     stop        label  duration
0     0.04  1611.04          ES0    1611.0
4     0.04  6022.04  task_GLOBAL    6022.0
1  1693.04  4357.04          BL0    2664.0
2  4710.04  6022.04          MC0    1312.0
3  6042.04  6973.04          BL1     931.0

[5 rows x 4 columns]

hardcoded_params.decoder_building_session_names: ['ES0', 'MC0', 'task_GLOBAL']
hardcoded_params.non_global_activity_session_names: ['ES0', 'MC0']
active_data_session_types_registered_classes_dict: {'dandi_nwb': <class 'neuropy.core.session.Formats.Specific.NWBDataSessionFormat.NWBDataSessionFormatRegisteredClass'>, 'kdiba': <class 'neuropy.core.session.Formats.Specific.KDibaOldDataSessionFormat.KDibaOldDataSessionFormatRegisteredClass'>, 'dandi_nwb_001754': <class 'neuropy.core.session.Formats.Specific.DANDI001754NWBDataSessionFormat.DANDI001754NWBDataSessionFormatRegisteredClass'>, 'bapun': <class 'neuropy.core.session.Formats.Specific.BapunDataSessionFormat.BapunDataSessionFormatRegisteredClass'>, 'rachel': <class 'neuropy.core.session.Formats.Specific.RachelDataSessionFormat.RachelDataSessionFormat'>}
WARN: already fixedup session epochs.
	done. new epochs: 
5 epochs
array([[0.04, 1611.04],
       [0.04, 6022.04],
       [1693.04, 4357.04],
       [4710.04, 6022.04],
       [6042.04

	 no change in time_variable_name. It will remain t_seconds.


Applying session filter named "task_GLOBAL"...


	 no change in time_variable_name. It will remain t_seconds.


Applying session filter named "MC0"...


	 no change in time_variable_name. It will remain t_seconds.


	hardcoded_params.grid_bin_bounds: ((0.0, 255.0), (0.0, 255.0))
beginning compute...
i: 0, active_epoch_names: ['ES0', 'MC0']
Performing perform_action_for_all_contexts with action EvaluationActions.EVALUATE_COMPUTATIONS on filtered_session with filter named "ES0"...
curr_active_computation_params.pf_params.computation_epochs: 1 epochs
array([[0.04, 1611.04]])

due to includelist, including only 3 out of 18 registered computation functions.
Performing _execute_computation_functions(...) with 3 registered_computation_functions...
Recomputing active_epoch_placefields1D... 	 done.
Recomputing active_epoch_placefields2D... 	 done.
Recomputing active_epoch_time_dependent_placefields... 	 done.
Recomputing active_epoch_time_dependent_placefields2D... 	 done.
_execute_computation_functions(...): 
	accumulated_errors: None
	computation_times: {'_perform_baseline_placefield_computation': datetime.datetime(2026, 7, 9, 16, 59, 57, 733371), '_perform_position_decoding_computation': datetime.dateti

In [12]:
epochs_decoding_time_bin_size = 0.250
new_decoder_dict, continuous_specific_decoded_results_dict, (contextual_pf2D_Decoder, contextual_pf2D_dict) = build_non_kdiba_directional_decoders(curr_active_pipeline, epochs_decoding_time_bin_size=epochs_decoding_time_bin_size)
list(new_decoder_dict.keys())


self.pf.ratemap.n_neurons == 0! (No Neurons!)


ValueError: self.pf.ratemap.n_neurons == 0! (No Neurons!)

In [ ]:
from pyphoplacecellanalysis.Pho2D.PyQtPlots.TimeSynchronizedPlotters.TimeSynchronizedSpikeRasterPlotter import TimeSynchronizedSpikeRasterPlotter

active_2d_plot = TimeSynchronizedSpikeRasterPlotter(curr_active_pipeline.sess, width=1400, height=500)
a_rect_item, an_interval_ds = build_proper_epoch_intervals(curr_active_pipeline=curr_active_pipeline, active_2d_plot=active_2d_plot)
active_2d_plot.show()


In [ ]:
curr_active_pipeline.save_pipeline(saving_mode=saving_mode, active_pickle_filename='loadedSessPickle.pkl')
print('Saved pipeline pickle to', basedir / 'loadedSessPickle.pkl')
print('Session export cache remains at', sess.filePrefix.parent)


In [ ]:
curr_active_pipeline.prepare_for_display()

# 🎨 2024-02-06 - Other Plotting

In [13]:
from pyphoplacecellanalysis.Pho2D.PyQtPlots.TimeSynchronizedPlotters.TimeSynchronizedPlacefieldsPlotter import TimeSynchronizedPlacefieldsPlotter

_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

#  Create a new `SpikeRaster2D` instance using `_display_spike_raster_pyqtplot_2D` and capture its outputs:
curr_active_pipeline.reload_default_display_functions()
curr_active_pipeline.prepare_for_display()

## `LauncherWidget`: GUI

In [ ]:
from pyphoplacecellanalysis.General.Pipeline.Stages.Display import DisplayFunctionItem
from pyphocorehelpers.gui.Qt.tree_helpers import find_tree_item_by_text
from pyphoplacecellanalysis.GUI.Qt.MainApplicationWindows.LauncherWidget.LauncherWidget import LauncherWidget

widget = LauncherWidget()
treeWidget = widget.mainTreeWidget # QTreeWidget
widget.build_for_pipeline(curr_active_pipeline=curr_active_pipeline)
widget.show()

setting icon to ":/Render/Icons/Icon/SimplePlot/Laps.png"...
setting icon to ":/Graphics/Icons/graphics/yellow_blue_plot_icon.png"...
setting icon to ":/Render/Icons/Icon/Pseudo2D.png"...
setting icon to ":/Icons/Icons/visualizations/template_1D_debugger.ico"...
setting icon to ":/Graphics/Icons/graphics/directional_track_template_pf1Ds.png"...
setting icon to ":/Icons/Icons/visualizations/rank_order_raster_debugger.ico"...
setting icon to ":/Graphics/Icons/graphics/Spikes.png"...
setting icon to ":/Render/Icons/Icon/Occupancy.png"...
setting icon to ":/Graphics/Icons/graphics/Spikes.png"...
setting icon to ":/Render/Icons/Icon/Occupancy.png"...
setting icon to ":/Render/Icons/Icon/HeatmapUgly.png"...
setting icon to ":/Render/Icons/Icon/Heatmap.png"...
setting icon to ":/Icons/Icons/SpikeRaster2DIcon.ico"...
setting icon to ":/Icons/Icons/SpikeRaster3DIcon.ico"...
setting icon to ":/Icons/Icons/SpikeRaster3D_VedoIcon.ico"...
setting icon to ":/Icons/Icons/InteractivePlaceCellDataExplo

on_selected_context_index_changed: 0, ES0, dandi_nwb_001754_Rat1_001754_ses-19980425T124500_ES0, dandi_nwb_001754_Rat1_001754_ses-19980425T124500_ES0
on_selected_context_changed(new_key: ES0, new_context: dandi_nwb_001754_Rat1_001754_ses-19980425T124500_ES0)
Uncaught Exception in slot


Traceback (most recent call last):
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoCoreHelpers\src\pyphocorehelpers\gui\Qt\ExceptionPrintingSlot.py", line 42, in wrapper
    return func(*args, **kwargs) # TODO 2025-01-05 - Seems to work in simple use case and allows calling via obustly and carefully modify `pyqtExceptionPrintingSlot` to enable calling decorated functions with their kwargs in additions to their *args. Currently calling a function decorated by `@pyqtExceptionPrintingSlot` like @commits_table_widget.py (1160-1162)  via its kwargs results in a runtime error: e.g. @src/pho_github_activity_viewer/widgets/commits_table_widget.py:1027fails while `self.set_grouping_enabled(False)` succeeds.
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoPlaceCellAnalysis\src\pyphoplacecellanalysis\GUI\Qt\MainApplicationWindows\LauncherWidget\LauncherWidget.py", line 398, in on_tree_item_double_clicked
    return self._perform_execute_display_function(a_fcn_name=a_fc

TrialByTrialActivity is not computed, computing it...
===>|> for filtered_session with filter named "ES0": Performing run_specific_computations_single_context(..., computation_functions_name_includelist=['pf_computation', 'pfdt_computation'])...
	run_specific_computations_single_context(including only 2 out of 18 registered computation functions): active_computation_functions: [<function PlacefieldComputations._perform_baseline_placefield_computation at 0x00000267E63E4B80>, <function PlacefieldComputations._perform_time_dependent_placefield_computation at 0x00000267E63E4D30>]...
Performing _execute_computation_functions(...) with 2 registered_computation_functions...
Executing [0/2]: <function PlacefieldComputations._perform_baseline_placefield_computation at 0x00000267E63E4B80>
Recomputing active_epoch_placefields1D... 	 done.
Recomputing active_epoch_placefields2D... 	 done.
Executing [1/2]: <function PlacefieldComputations._perform_time_dependent_placefield_computation at 0x00000267

Traceback (most recent call last):
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoCoreHelpers\src\pyphocorehelpers\gui\Qt\ExceptionPrintingSlot.py", line 42, in wrapper
    return func(*args, **kwargs) # TODO 2025-01-05 - Seems to work in simple use case and allows calling via obustly and carefully modify `pyqtExceptionPrintingSlot` to enable calling decorated functions with their kwargs in additions to their *args. Currently calling a function decorated by `@pyqtExceptionPrintingSlot` like @commits_table_widget.py (1160-1162)  via its kwargs results in a runtime error: e.g. @src/pho_github_activity_viewer/widgets/commits_table_widget.py:1027fails while `self.set_grouping_enabled(False)` succeeds.
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoPlaceCellAnalysis\src\pyphoplacecellanalysis\GUI\Qt\MainApplicationWindows\LauncherWidget\LauncherWidget.py", line 398, in on_tree_item_double_clicked
    return self._perform_execute_display_function(a_fcn_name=a_fc

...

Traceback (most recent call last):
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoCoreHelpers\src\pyphocorehelpers\gui\Qt\ExceptionPrintingSlot.py", line 42, in wrapper
    return func(*args, **kwargs) # TODO 2025-01-05 - Seems to work in simple use case and allows calling via obustly and carefully modify `pyqtExceptionPrintingSlot` to enable calling decorated functions with their kwargs in additions to their *args. Currently calling a function decorated by `@pyqtExceptionPrintingSlot` like @commits_table_widget.py (1160-1162)  via its kwargs results in a runtime error: e.g. @src/pho_github_activity_viewer/widgets/commits_table_widget.py:1027fails while `self.set_grouping_enabled(False)` succeeds.
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoPlaceCellAnalysis\src\pyphoplacecellanalysis\GUI\Qt\MainApplicationWindows\LauncherWidget\LauncherWidget.py", line 398, in on_tree_item_double_clicked
    return self._perform_execute_display_function(a_fcn_name=a_fc

Uncaught Exception in slot


Traceback (most recent call last):
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoCoreHelpers\src\pyphocorehelpers\gui\Qt\ExceptionPrintingSlot.py", line 42, in wrapper
    return func(*args, **kwargs) # TODO 2025-01-05 - Seems to work in simple use case and allows calling via obustly and carefully modify `pyqtExceptionPrintingSlot` to enable calling decorated functions with their kwargs in additions to their *args. Currently calling a function decorated by `@pyqtExceptionPrintingSlot` like @commits_table_widget.py (1160-1162)  via its kwargs results in a runtime error: e.g. @src/pho_github_activity_viewer/widgets/commits_table_widget.py:1027fails while `self.set_grouping_enabled(False)` succeeds.
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoPlaceCellAnalysis\src\pyphoplacecellanalysis\GUI\Qt\MainApplicationWindows\LauncherWidget\LauncherWidget.py", line 398, in on_tree_item_double_clicked
    return self._perform_execute_display_function(a_fcn_name=a_fc

Uncaught Exception in slot


Traceback (most recent call last):
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoCoreHelpers\src\pyphocorehelpers\gui\Qt\ExceptionPrintingSlot.py", line 42, in wrapper
    return func(*args, **kwargs) # TODO 2025-01-05 - Seems to work in simple use case and allows calling via obustly and carefully modify `pyqtExceptionPrintingSlot` to enable calling decorated functions with their kwargs in additions to their *args. Currently calling a function decorated by `@pyqtExceptionPrintingSlot` like @commits_table_widget.py (1160-1162)  via its kwargs results in a runtime error: e.g. @src/pho_github_activity_viewer/widgets/commits_table_widget.py:1027fails while `self.set_grouping_enabled(False)` succeeds.
  File "H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\pyPhoPlaceCellAnalysis\src\pyphoplacecellanalysis\GUI\Qt\MainApplicationWindows\LauncherWidget\LauncherWidget.py", line 398, in on_tree_item_double_clicked
    return self._perform_execute_display_function(a_fcn_name=a_fc

In [ ]:
session_id_str: str = curr_active_pipeline.get_complete_session_identifier_string()
widget.setWindowTitle(f'Spike3D Launcher: {session_id_str}')
treeWidget.root
# curr_active_pipeline.get_session_additional_parameters_context()
# curr_active_pipeline.get_complete_session_context()

## ✅ 2025-09-19 - Clean programmmatic figure outputs 

In [ ]:
from pyphocorehelpers.plotting.figure_management import PhoActiveFigureManager2D, capture_new_figures_decorator
fig_man = PhoActiveFigureManager2D(name=f'fig_man') # Initialize a new figure manager
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.DockAreaWrapper import DockAreaWrapper
from pyphoplacecellanalysis.General.Mixins.ExportHelpers import programmatic_render_to_file, programmatic_display_to_PDF, extract_figures_from_display_function_output
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters
from neuropy.core.session.Formats.Specific.NWBDataSessionFormat import NWBDataSessionFormatRegisteredClass


In [ ]:

hardcoded_params: HardcodedProcessingParameters = NWBDataSessionFormatRegisteredClass._get_session_specific_parameters(session_context=curr_active_pipeline.get_session_context())
# hardcoded_params

# hardcoded_params.decoder_building_session_names
# hardcoded_params.non_global_activity_session_names

fig_man.close_all()

# subset_includelist = ['maze1', 'maze2', 'maze_GLOBAL'] # Day5TwoNovel
# subset_includelist = ['roam', 'sprinkle'] # Day4

In [ ]:

# subset_includelist = hardcoded_params.decoder_building_session_names
subset_includelist = None
print(f'subset_includelist: {subset_includelist}')

In [ ]:
display_fn_kwargs = dict(subplots=(None, 9),
    fig_column_width=None,   # key fix — uses data aspect ratio for width
    fig_row_height=1.0,
    resolution_multiplier=1.0,
)

# display_fn_kwargs = dict(subplots=(None, 5))

# _out = dict()
# _out['_display_2d_placefield_result_plot_ratemaps_2D'] = curr_active_pipeline.display(display_function='_display_2d_placefield_result_plot_ratemaps_2D', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatS',session_name='Day5TwoNovel',filter_name='maze1'), **display_fn_kwargs) # _display_2d_placefield_result_plot_ratemaps_2D
# _out['_display_2d_placefield_result_plot_ratemaps_2D'] = curr_active_pipeline.display(display_function='_display_2d_placefield_result_plot_ratemaps_2D', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatS',session_name='Day5TwoNovel',filter_name='maze2'), **display_fn_kwargs) # _display_2d_placefield_result_plot_ratemaps_2D


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_2d_placefield_result_plot_ratemaps_2D', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True, **display_fn_kwargs)


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_2d_placefield_occupancy', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True)


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_1d_placefields', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True, **display_fn_kwargs)


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_1d_placefield_validations', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True, **display_fn_kwargs)